# Control for arm 7': baseline trained for as many gradient steps as the joint arm

Arm 7' (HBFMID + FracAtlas, 100 epochs) trains on 933 + 717 images per epoch, so it takes about 1.77x the gradient steps of the baseline (933 images, 100 epochs). This control trains the plain baseline for 177 epochs (same number of steps as arm 7'), seed 42, 5 folds, so that the effect of the FracAtlas images can be separated from the effect of longer training.
Fixed-epoch protocol, last.pt. 5 runs, about 45 min each, estimated ~2 h on T4 x2. Arm tag `ep177`. Settings: GPU T4 x2, Internet ON, HBFMID dataset only.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/nithoukhan1/hbfmid.git"
REPO = "/kaggle/working/hbfmid"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.215"], check=True)
os.environ["WANDB_MODE"] = "disabled"
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2, then restart."
print("repo commit:", subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
DATA_ROOT = "/kaggle/input/datasets/nithoukhan2/hbfmid/Human Bone Fractures Multi-modal Image Dataset (HBFMID)/Bone Fractures Detection"
assert os.path.isdir(DATA_ROOT), DATA_ROOT
SMOKE = False
MODEL = "yolov8s.pt"
SEEDS = [42]
if SMOKE:
    EPOCHS, OUT, FOLDS = 2, "/kaggle/working/results/ctrl_ep177_smoke", [0]
else:
    EPOCHS, OUT, FOLDS = 177, "/kaggle/working/results/ctrl_ep177", [0, 1, 2, 3, 4]

def make_cmd(fold, seed):
    def f(gpu):
        return [sys.executable, f"{REPO}/scripts/training/run_baseline.py", "--mode", "clean", "--fold", str(fold), "--seed", str(seed),
                "--model", MODEL, "--epochs", str(EPOCHS), "--fixed-epochs", "--device", str(gpu), "--data-root", DATA_ROOT,
                "--splits-dir", f"{REPO}/data_splits", "--out", OUT, "--arm", "ep177"]
    return f
TASKS = [(f"clean_f{f}_s{s}_ep177", make_cmd(f, s), None) for f in FOLDS for s in SEEDS]
print(len(TASKS), "runs, epochs =", EPOCHS, "->", OUT)

In [ ]:
import queue, threading, time
os.makedirs(f"{OUT}/logs", exist_ok=True)
gpus = list(range(torch.cuda.device_count()))
q = queue.Queue()
for t in TASKS:
    q.put(t)
failed = []

def worker(gpu):
    while True:
        try:
            name, cmd_fn, needs = q.get_nowait()
        except queue.Empty:
            return
        while needs and not os.path.exists(needs):          # e.g. arm 6 waits for the FracAtlas pre-trained checkpoint
            if any(n.startswith("fa_pre_") for n in failed):
                break
            time.sleep(20)
        t0 = time.time()
        with open(f"{OUT}/logs/{name}.log", "w") as f:
            rc = subprocess.run(cmd_fn(gpu), stdout=f, stderr=subprocess.STDOUT).returncode
        print(f"[GPU {gpu}] {name}: {'OK' if rc == 0 else 'FAILED rc=' + str(rc)} in {(time.time()-t0)/60:.1f} min", flush=True)
        if rc != 0:
            failed.append(name)

threads = [threading.Thread(target=worker, args=(g,)) for g in gpus]
[t.start() for t in threads]; [t.join() for t in threads]
print("failed runs:", failed or "none")
for n in failed:
    print("\n==== tail of", n)
    print("".join(open(f"{OUT}/logs/{n}.log").readlines()[-25:]))

In [ ]:
import shutil
BASE = [f"{REPO}/results/controls", f"{REPO}/results/baseline_fixed"]
subprocess.run([sys.executable, f"{REPO}/scripts/evaluation/aggregate_arm.py", "--name", "arm_ep177", "--runs", OUT, "--arm-tag", "ep177", "--out", OUT])
subprocess.run([sys.executable, f"{REPO}/scripts/evaluation/compare_arms.py", "--data-root", DATA_ROOT, "--baseline-runs", *BASE,
                "--arm-runs", OUT, "--arm-name", "ep177", "--arm-tag", "ep177", "--out", f"{OUT}/ep177_vs_baseline"])
shutil.make_archive("/kaggle/working/ctrl_ep177_results", "zip", OUT)
print(sorted(os.listdir(OUT)))
if failed:
    print("\nWARNING: failed runs:", failed)

After the run: download `ctrl_ep177_results.zip`, unzip once into `results/ctrl_ep177/`, commit and push.